# NB03 — The Oxygen Attack on Ethylene: Why Transition States Are Multireference

The reaction of atomic oxygen O(³P) with ethylene provides a prototypical example where the electronic structure changes qualitatively along a reaction coordinate. In the transition-state region, the system develops pronounced biradical character, and single-reference methods become inadequate.

## The reaction
We consider the attack of atomic oxygen O(³P) on the C=C double bond of ethylene.
$$
\mathrm{O}(^3P) + \mathrm{C_2H_4} \rightarrow \text{triplet biradical intermediate}
$$

![Reaction scheme](figures/o3p_ethylene_scheme.svg)

As the oxygen atom approaches the π bond, electron pairing in the double bond is progressively weakened. The system evolves from a closed-shell π-bonded structure toward a configuration in which two unpaired electrons are distributed over the O–C–C framework.

In this regime, multiple electronic configurations become nearly degenerate, and a single determinant is no longer sufficient to describe the wavefunction.

We will build the potential energy surface step by step. Look at the curves before reading the explanation.


## 0. Setup (run this first)
This cell loads the required libraries and configures paths. It is not part of the scientific content — run it once before proceeding.


In [ ]:
import sys
sys.path.insert(0, '../tools')

import shutil
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

from utils import (
    ORCA, NPROCS, HARTREE_TO_KJMOL,
    setup_workdir, run_orca,
    get_energy, get_nevpt2_energy,
    get_distance, terminated_normally,
    get_no_occupations, plot_orbital, show_orbital
)
from qctools import load_xyz_as_traj, build_xyz_trajectory

# O-C1 distance extractor — atoms 0 and 2 in the xyz files
get_oc_distance = lambda f: get_distance(f, 0, 2)

# Include %pal block only when more than one core is available
pal_block = f'%pal nprocs {NPROCS} end\n\n' if NPROCS > 1 else ''

print(f'ORCA:   {ORCA}')
print(f'NPROCS: {NPROCS}')

## 1. Working directory
Calculation files will be written to `ethylene/`. Set `FORCE_CLEAN = True` to delete and restart from scratch — useful if you want to rerun everything cleanly.


In [ ]:
WORKDIR = 'ethylene'
FORCE_CLEAN = False  # set True to start from scratch

if FORCE_CLEAN and Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
    print(f'Removed {WORKDIR}/')

work_dir = setup_workdir(WORKDIR)
print(f'Working in: {work_dir}')

## 2. Starting geometry

O at ~3.5 Å from C1, ethylene in its equilibrium geometry.
Charge 0, multiplicity 3 (triplet O(³P) + singlet ethylene).


In [ ]:
# Ethylene heavy-atom positions (frozen throughout the diagnostic scan)
C1 = np.array([ 0.000,  0.000,  0.000])
C2 = np.array([ 1.335,  0.000,  0.000])
O0 = np.array([-0.517,  0.000,  1.932])   # starting position, R = 3.5 Å

ts_geometry = """
C    0.000000    0.000000    0.000000
C    1.335000    0.000000    0.000000
O   -0.517000    0.000000    1.932000
H   -0.585000    0.925000   -0.150000
H   -0.585000   -0.925000   -0.150000
H    1.920000    0.925000    0.050000
H    1.920000   -0.925000    0.050000
"""
print(ts_geometry)

The starting structure has O(³P) at 3.5 Å from C1, well outside bonding distance.
Ethylene is planar and undistorted — the π bond is intact.


In [ ]:
import nglview
from ase import Atoms

# Build ASE Atoms from the starting geometry for display
symbols = ['C', 'C', 'O', 'H', 'H', 'H', 'H']
positions = [
    [ 0.000000,  0.000000,  0.000000],
    [ 1.335000,  0.000000,  0.000000],
    [-0.517000,  0.000000,  1.932000],
    [-0.585000,  0.925000, -0.150000],
    [-0.585000, -0.925000, -0.150000],
    [ 1.920000,  0.925000,  0.050000],
    [ 1.920000, -0.925000,  0.050000],
]
mol = Atoms(symbols=symbols, positions=positions)
view = nglview.show_ase(mol)
view._set_size('500px', '400px')
view.clear_representations()
view.add_ball_and_stick()
view.parameters = dict(backgroundColor='white')
view

## 3. Active space selection

CASSCF requires us to choose a subset of orbitals — the *active space* — in which the wavefunction is described as a superposition of electron configurations. The choice matters: too small an active space misses the relevant physics; too large is computationally impractical.

The guiding principle is to include the orbitals where electrons are *neither clearly bonded nor clearly absent* along the reaction coordinate. For this reaction those are:

| Orbital at R=2.2 Å | Orbital at R=1.5 Å | Physical evolution |
|-------------------|-------------------|--------------------|
| π (C=C), symmetric, occ ≈ 1.91 | π/O mixed bonding, occ ≈ 2.0 | π evolves into a 3-centre π–O interaction as O approaches C1 |
| π* (C=C), antisymmetric, occ ≈ 0.09 | π*/C2 radical precursor, occ ≈ 0.01 | π* gains C2 radical character as C=C bond begins to break |
| p (O, in-plane) SOMO, occ ≈ 1.0 | σ (O–C) bond, occ ≈ 2.0 | O lone pair rotates into reaction plane and forms the O–C σ bond |
| p (O, out-of-plane) lone pair, occ ≈ 2.0 | p (O) SOMO, occ ≈ 1.0 | Doubly occupied lone pair becomes the persistent O-centred SOMO |
| O p / σ mix SOMO, occ ≈ 1.0 | C2 radical SOMO, occ ≈ 1.0 | SOMO migrates from oxygen to C2, defining the carbon radical centre |
| σ* (O–C), near-empty, occ ≈ 0.003 | σ* (O–C), near-empty, occ ≈ 0.003 | Spectating antibond — occupation unchanged throughout |

This gives **6 electrons in 6 orbitals** — CASSCF(6,6).

Note that σ* remains near-empty throughout and could arguably be omitted from a minimal active space.
It is retained here for completeness and because the σ/σ* pair should always be included together.

We visualise these orbitals at two geometries: R=2.2 Å (entrance-channel complex, before the O–C bond is formed)
and R=1.5 Å (biradical-forming region, O–C bond nearly complete). Orbitals are paired by physical
character rather than MO index — the index ordering can change as the reaction proceeds.
The geometries are unrelaxed — ethylene frozen, O moved linearly toward C1.


In [ ]:
# Generate diagnostic geometries at R=2.2 Å and R=1.5 Å
# Ethylene frozen; O moved linearly toward C1
vec  = C1 - O0
R0   = np.linalg.norm(vec)
uvec = vec / R0

ethylene_tail = """H   -0.585000    0.925000   -0.150000
H   -0.585000   -0.925000   -0.150000
H    1.920000    0.925000    0.050000
H    1.920000   -0.925000    0.050000"""

def make_geom(R):
    O_pos = O0 + (R0 - R) * uvec
    return f"""C    {C1[0]:.6f}    {C1[1]:.6f}    {C1[2]:.6f}
C    {C2[0]:.6f}    {C2[1]:.6f}    {C2[2]:.6f}
O    {O_pos[0]:.6f}    {O_pos[1]:.6f}    {O_pos[2]:.6f}
{ethylene_tail}""", O_pos

geom_barrier, O_barrier = make_geom(2.2)
geom_product, O_product = make_geom(1.5)

print(f'R = 2.2 Å  ->  O at ({O_barrier[0]:.3f}, {O_barrier[1]:.3f}, {O_barrier[2]:.3f})')
print(f'R = 1.5 Å  ->  O at ({O_product[0]:.3f}, {O_product[1]:.3f}, {O_product[2]:.3f})')

In [ ]:
# CASSCF(6,6) single points at R=2.2 Å and R=1.5 Å
# SwitchStep 0.0 prevents ORCA from swapping active/inactive orbitals
# during optimization, keeping the active space identity stable.
tag_barrier = 'noon_diag_2p2'
tag_product = 'noon_diag_1p5'

for tag, geom, R_label in [
        (tag_barrier, geom_barrier, '2.2'),
        (tag_product, geom_product, '1.5'),
]:
    inp = f"""! CASSCF cc-pVDZ TightSCF

{pal_block}%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
  SwitchStep 0.0
end

* xyz 0 3
{geom}
*
"""
    print(f'Running CASSCF(6,6) at R={R_label} A ...')
    run_orca(tag, inp, work_dir)
print('Done.')

The six CASSCF(6,6) natural orbitals paired by physical character across the two geometries.
Blue and red lobes show opposite wavefunction phases.

The central story is the evolution of two SOMOs: the O in-plane p orbital (singly occupied at R=2.2 Å)
transforms into the O–C σ bond, while a second O-centred SOMO migrates to C2, producing the
triplet biradical. The π system evolves from a symmetric C=C bond toward a 3-centre π–O interaction.

Orbitals are matched by physical character, not by MO index, which reorders along the path.


In [ ]:
# Active space orbital gallery — paired by physical character, not MO index
# Each row: (MO index at R=2.2, label, MO index at R=1.5, label)
import py3Dmol
from IPython.display import HTML

pairs = [
    (10, 'pi(C=C), symmetric',        10, 'pi/O mixed bonding'         ),
    (13, 'pi*(C=C), antisymmetric',   13, 'pi*/C2 radical precursor'   ),
    (11, 'p(O, in-plane) SOMO',        9, 'sigma(O-C) bond'            ),
    ( 9, 'p(O, out-of-plane) lone pair',12,'p(O) SOMO'                 ),
    (12, 'O p/sigma mix SOMO',         11, 'C2 radical SOMO'           ),
    (14, 'sigma*(O-C), spectating',    14, 'sigma*(O-C), spectating'   ),
]

occs_barrier = get_no_occupations(work_dir / f'{tag_barrier}.out')[-6:]
occs_product = get_no_occupations(work_dir / f'{tag_product}.out')[-6:]

for mo_b, lbl_b, mo_p, lbl_p in pairs:
    cube_b = plot_orbital(tag_barrier, mo_index=mo_b, work_dir=work_dir)
    cube_p = plot_orbital(tag_product,  mo_index=mo_p, work_dir=work_dir)

    occ_b = occs_barrier[mo_b - 9]
    occ_p = occs_product[mo_p - 9]
    changed = lbl_b != lbl_p
    note = '&nbsp; &#9888; character change' if changed else ''

    display(HTML(f"""
    <div style="display:flex; width:800px; font-family:sans-serif; font-size:13px;
                margin-top:16px; margin-bottom:2px;">
      <div style="width:400px; text-align:center; color:#2980b9;">
        <b>{lbl_b}</b>&nbsp; R=2.2 &#8491;&nbsp; occ={occ_b:.3f}
      </div>
      <div style="width:400px; text-align:center; color:#e67e22;">
        <b>{lbl_p}</b>&nbsp; R=1.5 &#8491;&nbsp; occ={occ_p:.3f}{note}
      </div>
    </div>
    """))

    view = py3Dmol.view(width=800, height=280, linked=False,
                        viewergrid=(1, 2))
    for col, cube in [(0, cube_b), (1, cube_p)]:
        view.addVolumetricData(cube, 'cube',
                               {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                               viewer=(0, col))
        view.addVolumetricData(cube, 'cube',
                               {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                               viewer=(0, col))
        view.addModel(cube, 'cube', viewer=(0, col))
        view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.1}},
                      viewer=(0, col))
    view.zoomTo()
    view.show()

<details>
<summary>Active space selection — practical criteria and orbital evolution</summary>

The labels above are assigned from Loewdin orbital compositions in the ORCA output and confirmed
against established multireference studies of O(³P) + ethylene.

The key physical evolution along the reaction coordinate:

- The O in-plane p SOMO (occ ≈ 1.0 at R=2.2 Å) rotates into the reaction plane as O approaches
  C1 and becomes the O–C σ bond (occ ≈ 2.0 at R=1.5 Å). This is the lone pair → bond transformation.
- The O out-of-plane lone pair (occ ≈ 2.0 at R=2.2 Å) loses one electron as the σ bond forms,
  becoming the persistent O-centred SOMO (occ ≈ 1.0) of the product biradical.
- A third O-centred orbital (occ ≈ 1.0 at R=2.2 Å) migrates to C2 as the C=C bond breaks,
  forming the C2-centred SOMO of the biradical. This orbital reordering is an avoided crossing
  within the active space — normal and expected along a bond-forming coordinate.
- The π system evolves from a symmetric C1–C2 bond toward a 3-centre π–O interaction,
  reflecting the developing sp³ character at C1.
- σ* remains spectating (occ ≈ 0.003) throughout. It is retained for the completeness of the
  σ/σ* pair — omitting one half would introduce a spurious asymmetry — but a CASSCF(4,4)
  omitting both σ and σ* would give a qualitatively similar barrier.

`SwitchStep 0.0` prevents ORCA from swapping active orbitals with inactive ones during optimization.
It does not prevent reordering within the active space — that is a physical feature, not a numerical artefact.

</details>


## 4. CASSCF relaxed scan — custom R grid

Active space: 6 electrons in 6 orbitals — the π/π* of ethylene, the two singly occupied p orbitals on oxygen, and the nascent O–C σ/σ* pair.

Rather than a linear grid, we use a **non-uniform spacing** chosen to put more points where the energy changes rapidly (barrier region, R=1.2–2.5 Å) and fewer in the flat long-range tail (R=2.5–3.5 Å).

Each point is a constrained geometry optimisation: the O–C1 distance is fixed at the target R while all other degrees of freedom are relaxed. The O–C bond is constrained using `{B 0 2 R C}` in the `%geom Constraints` block.

Points are computed inward (large R → small R) with the converged wavefunction passed forward at each step via `MORead` — the same warm-starting that ORCA uses internally for its scan jobs.

`MaxIter 200` gives the CASSCF optimizer enough room to converge near the barrier.


In [ ]:
import threading, time

# Non-uniform R grid: coarse in flat tail, fine through barrier and product
R_grid = [
    3.50, 3.20, 2.90,          # tail: 3 points, ~0.3 A spacing
    2.60, 2.40, 2.20,          # approach: 3 points, ~0.2 A spacing
    2.05, 1.90, 1.80, 1.70,   # barrier: 4 points, ~0.1 A spacing
    1.60, 1.50, 1.40, 1.30,   # product region: 4 points, ~0.1 A spacing
    1.22,                      # near minimum
]  # 15 points total

casscf_results = []   # list of (R, E, xyz_path)
prev_gbw  = None
prev_geom = ts_geometry  # start from input geometry at R=3.5 A

for R in R_grid:
    tag = f'casscf_R{R:.2f}'.replace('.', 'p')
    gbw_out = (work_dir / tag).with_suffix('.gbw').resolve()
    xyz_out = work_dir / f'{tag}.xyz'

    moread = f'! MORead\n%moinp "{prev_gbw}"\n\n' if prev_gbw else ''

    inp = f"""{moread}! CASSCF cc-pVDZ TightSCF Opt

{pal_block}%geom
  Constraints
    {{B 0 2 {R:.3f} C}}
  end
end

%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
  MaxIter 200
end

* xyz 0 3
{prev_geom}
*
"""

    print(f'  R = {R:.2f} A ...', end=' ', flush=True)
    outfile = run_orca(tag, inp, work_dir)
    E = get_energy(outfile)
    ok = terminated_normally(outfile)
    print(f'E = {E:.6f} Eh  ({"OK" if ok else "FAILED"})', flush=True)

    casscf_results.append((R, E))

    # Update warm-start: use relaxed geometry for next step
    if gbw_out.exists():
        prev_gbw = gbw_out
    # Read relaxed geometry from ORCA output for next step
    try:
        text = outfile.read_text()
        # Find last 'CARTESIAN COORDINATES (A.U.)' block
        idx = text.rfind('CARTESIAN COORDINATES (ANGSTROEM)')
        if idx >= 0:
            block = text[idx:].splitlines()
            geom_lines = []
            for line in block[2:]:
                parts = line.split()
                if len(parts) == 4 and parts[0].isalpha():
                    geom_lines.append(f'{parts[0]:2s}  {float(parts[1]):12.6f}  {float(parts[2]):12.6f}  {float(parts[3]):12.6f}')
                elif geom_lines:
                    break
            if geom_lines:
                prev_geom = '\n'.join(geom_lines)
    except Exception:
        pass  # keep previous geometry on failure

print('\nAll CASSCF points done.')
R_casscf_arr = np.array([r for r, e in casscf_results])
E_casscf_arr = np.array([e for r, e in casscf_results])

## 5. NEVPT2 single points

NEVPT2 adds dynamic correlation on top of the CASSCF reference.
We use the `.gbw` file from each constrained optimisation as the orbital guess via `MORead`.


In [ ]:
nevpt2_results = []

for R, E_cas in casscf_results:
    cas_tag = f'casscf_R{R:.2f}'.replace('.', 'p')
    gbw_file = (work_dir / cas_tag).with_suffix('.gbw').resolve()
    xyz_file = work_dir / f'{cas_tag}.xyz'
    tag = f'nevpt2_R{R:.2f}'.replace('.', 'p')

    # Read geometry from CASSCF output
    if xyz_file.exists():
        lines = xyz_file.read_text().splitlines()
        geom = '\n'.join(lines[2:])
    else:
        # Fall back to reading from ORCA output
        outfile = work_dir / f'{cas_tag}.out'
        text = outfile.read_text()
        idx = text.rfind('CARTESIAN COORDINATES (ANGSTROEM)')
        block = text[idx:].splitlines()
        geom_lines = []
        for line in block[2:]:
            parts = line.split()
            if len(parts) == 4 and parts[0].isalpha():
                geom_lines.append(f'{parts[0]:2s}  {float(parts[1]):12.6f}  {float(parts[2]):12.6f}  {float(parts[3]):12.6f}')
            elif geom_lines:
                break
        geom = '\n'.join(geom_lines)

    inp = f"""! NEVPT2 cc-pVDZ TightSCF MORead

{pal_block}%moinp "{gbw_file}"

%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
end

* xyz 0 3
{geom}
*
"""
    print(f'  R = {R:.2f} A ...', end=' ', flush=True)
    outfile = run_orca(tag, inp, work_dir)
    E = get_nevpt2_energy(outfile)
    print(f'E = {E:.6f} Eh', flush=True)
    nevpt2_results.append((R, E))

print('All NEVPT2 points done.')
R_nevpt2_arr = np.array([r for r, e in nevpt2_results])
E_nevpt2_arr = np.array([e for r, e in nevpt2_results])

## 6. B3LYP single points on CASSCF geometries

B3LYP is evaluated at the same geometries as the CASSCF scan — this isolates the effect
of the method from any geometry differences.

We run **outward** (small R → large R), starting from the product side where the triplet
wavefunction is well-defined. The warm-start chain then carries the converged solution
through the barrier region from below, which is more robust than approaching from the
reactant side where the wavefunction becomes unstable.

`STABPerform true` checks wavefunction stability. Any remaining gaps indicate geometries
where even the warm-started solution is intrinsically unstable — these appear as gaps in the plot.


In [ ]:
# B3LYP single points with warm-starting, running outward (small R -> large R)
# Starting from the product side where the triplet is well-defined gives
# better convergence through the barrier region than starting from reactants.
b3lyp_results = []
prev_gbw = None

for R, E_cas in sorted(casscf_results, key=lambda x: x[0]):  # small R first (outward)
    cas_tag  = f'casscf_R{R:.2f}'.replace('.', 'p')
    xyz_file = work_dir / f'{cas_tag}.xyz'
    tag      = f'b3lyp_R{R:.2f}'.replace('.', 'p')
    gbw_out  = (work_dir / tag).with_suffix('.gbw').resolve()

    # Read relaxed geometry
    if xyz_file.exists():
        lines = xyz_file.read_text().splitlines()
        geom  = '\n'.join(lines[2:])
    else:
        text  = (work_dir / f'{cas_tag}.out').read_text()
        idx   = text.rfind('CARTESIAN COORDINATES (ANGSTROEM)')
        block = text[idx:].splitlines()
        gl    = []
        for line in block[2:]:
            parts = line.split()
            if len(parts) == 4 and parts[0].isalpha():
                gl.append(f'{parts[0]:2s}  {float(parts[1]):12.6f}  {float(parts[2]):12.6f}  {float(parts[3]):12.6f}')
            elif gl:
                break
        geom = '\n'.join(gl)

    moread_block = f'! MORead\n%moinp "{prev_gbw}"\n\n' if prev_gbw else ''
    pal = f'%pal nprocs {NPROCS} end\n\n' if NPROCS > 1 else ''

    inp = f"""{moread_block}! B3LYP cc-pVDZ TightSCF SlowConv

{pal}%scf
  MaxIter 500
  STABPerform true
end

* xyz 0 3
{geom}
*
"""
    print(f'  R = {R:.2f} A ...', end=' ', flush=True)
    outfile = run_orca(tag, inp, work_dir)
    E = get_energy(outfile)
    print(f'E = {E:.6f} Eh', flush=True)
    b3lyp_results.append((R, E))
    if gbw_out.exists():
        prev_gbw = gbw_out

b3lyp_results.sort(key=lambda x: -x[0])  # large R first
R_b3lyp_arr = np.array([r for r, e in b3lyp_results])
E_b3lyp_arr = np.array([e for r, e in b3lyp_results])
print(f'B3LYP done. Convergence failures: {np.isnan(E_b3lyp_arr).sum()}')

## 7. Collect and normalise

All three methods share the same CASSCF-relaxed geometries at the custom R grid points.
Energies are normalised to zero at the reactant asymptote (largest R = 3.5 Å).


In [ ]:
# All arrays already sorted large R first
R_all = R_casscf_arr   # same R values for all three methods

# Normalise to largest R (index 0)
E_casscf_rel = (E_casscf_arr - E_casscf_arr[0]) * HARTREE_TO_KJMOL
E_nevpt2_rel = (E_nevpt2_arr - E_nevpt2_arr[0]) * HARTREE_TO_KJMOL

b3lyp_ref   = E_b3lyp_arr[0] if np.isfinite(E_b3lyp_arr[0]) else \
               E_b3lyp_arr[np.isfinite(E_b3lyp_arr)][0]
E_b3lyp_rel = np.where(
    np.isfinite(E_b3lyp_arr),
    (E_b3lyp_arr - b3lyp_ref) * HARTREE_TO_KJMOL,
    np.nan
)

print(f'R range: {R_all.min():.2f} - {R_all.max():.2f} A  ({len(R_all)} points)')
print(f'B3LYP convergence failures: {np.isnan(E_b3lyp_arr).sum()}')

## 8. Energy profile

Three methods, same geometries, same basis set. Look at the curves first — then read the table.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(R_all, E_casscf_rel, 'o--', color='steelblue',   label='CASSCF(6,6)')
ax.plot(R_all, E_nevpt2_rel, 'o-',  color='darkorange',  label='CASSCF + NEVPT2')
ax.plot(R_all, E_b3lyp_rel,  'o-',  color='forestgreen', label='B3LYP')

ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('R(O\u2013C1) [\u00c5]', fontsize=12)
ax.set_ylabel('Relative energy [kJ/mol]', fontsize=12)
ax.set_title('O(\u00b3P) attack on ethylene: B3LYP vs CASSCF vs NEVPT2', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(work_dir / 'final_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', work_dir / 'final_profile.png')

| Method | Barrier? | Why? |
|--------|----------|------|
| B3LYP | None | Single-reference — cannot describe the open-shell biradical character near the TS |
| CASSCF(6,6) | Yes, ~90 kJ/mol | Multireference, but missing dynamic correlation — overestimates the barrier |
| CASSCF + NEVPT2 | ~50 kJ/mol | Multireference + dynamic correlation — best estimate |

The B3LYP gaps are geometries where the single-reference wavefunction was unstable and ORCA aborted.
The difference between CASSCF and NEVPT2 quantifies the contribution of dynamic correlation to the barrier height.


## 9. Build animation trajectory


In [ ]:
traj_path = work_dir / 'full_path.xyz'
all_xyz_sorted = build_xyz_trajectory(
    list(work_dir.glob('casscf_R*.xyz')),
    traj_path,
    key=get_oc_distance,
    reverse=True
)

# R and NEVPT2 energies in trajectory order (large R first)
R_frames      = np.array([get_oc_distance(f) for f in all_xyz_sorted])
nevpt2_dict   = dict(nevpt2_results)
nevpt2_frames = np.array([nevpt2_dict.get(round(get_oc_distance(f), 2), float('nan'))
                           for f in all_xyz_sorted])
E_frames = (nevpt2_frames - nevpt2_frames[0]) * HARTREE_TO_KJMOL

print(f'Trajectory: {len(R_frames)} frames, R = {R_frames.max():.2f} \u2192 {R_frames.min():.2f} \u00c5')

## 10. Interactive trajectory viewer

Step through frames using the nglview player controls (hover over the molecule).
The red dot on the energy curve tracks the current geometry.

Things to look for:
- Pyramidalization of C1 as O approaches
- C=C bond lengthening through the barrier region
- Geometry of the triplet biradical intermediate at short R


In [ ]:
import nglview
%matplotlib widget

traj = load_xyz_as_traj(str(traj_path), silent=True)
view = nglview.show_asetraj(traj)
view._set_size('400px', '350px')

plt.ioff()
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(R_frames, E_frames, 'o-', color='darkorange', label='NEVPT2')
marker, = ax.plot([R_frames[0]], [E_frames[0]], 'ro', ms=10)
ax.set_xlabel('R(O\u2013C1) [\u00c5]')
ax.set_ylabel('Relative energy [kJ/mol]')
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
fig.canvas.layout = widgets.Layout(width='400px', height='350px')

def on_frame_change(change):
    i = change['new']
    marker.set_data([R_frames[i]], [E_frames[i]])
    fig.canvas.draw_idle()

view.observe(on_frame_change, names=['frame'])

panel = widgets.HBox([view, fig.canvas])
display(panel)
plt.ion()

## 11. Questions for Students

1. **Why does B3LYP predict no barrier?**  
   Think about the electronic structure of O(³P) and what happens to the spin as it approaches the π system.

2. **Why does B3LYP fail to converge at several geometries in the approach region?**  
   What does a negative stability eigenvalue mean physically?

3. **Why does CASSCF overestimate the barrier?**  
   What kind of electron correlation is CASSCF missing? What does NEVPT2 add?

4. **What is the active space (6,6) capturing here?**  
   Look at the orbital gallery in section 3. Which orbitals have fractional occupations, and what does that tell you about where the multiconfigurational character is concentrated?

5. **How does the geometry change along the reaction path?**  
   Use the animation to identify when pyramidalization of C1 begins.
   What does this tell us about the timing of bond formation?

6. **How does spin density change along the reaction path?**  
   Where does the unpaired spin end up in the product?
   What kind of intermediate is formed?
